In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

print("--- SCRIPT 1: INICIALIZACIÓN Y CAPA BRONZE ---")
spark = SparkSession.builder \
    .appName("NYC_GreenTaxi_Pipeline") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# Lectura de datos crudos (apuntando a la carpeta work/)
df_bronze_trips = spark.read.parquet("work/green_tripdata_*.parquet")
df_bronze_zones = spark.read.csv("work/taxi-zone-lookup.csv", header=True, inferSchema=True)

print(f"Registros crudos cargados en Bronze: {df_bronze_trips.count()}")

--- SCRIPT 1: INICIALIZACIÓN Y CAPA BRONZE ---
Registros crudos cargados en Bronze: 1251593


In [3]:
print("--- SCRIPT 2: ANÁLISIS EXPLORATORIO (EDA) EN RAW DATA ---")
df_bronze_trips.select("trip_distance", "fare_amount", "passenger_count").describe().show()

--- SCRIPT 2: ANÁLISIS EXPLORATORIO (EDA) EN RAW DATA ---
+-------+------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|   passenger_count|
+-------+------------------+------------------+------------------+
|  count|           1251593|           1251593|           1177385|
|   mean|17.967127684478996|18.229632548279376|1.3047142608407616|
| stddev|1091.8275730661112| 17.57877049959397|0.9601547219433944|
|    min|               0.0|            -470.6|                 0|
|    max|         262315.94|            1422.6|                 9|
+-------+------------------+------------------+------------------+



In [4]:
print("--- SCRIPT 3: TRANSFORMACIÓN Y CAPA SILVER ---")

# 1. Filtro de outliers y nulos (Veracidad)
df_silver = df_bronze_trips.filter(
    (F.col("trip_distance") > 0) & 
    (F.col("fare_amount") > 0) & 
    (F.col("passenger_count") > 0)
).dropna(subset=["PULocationID", "DOLocationID"])

# 2. Extracción de variables temporales (Transformación)
df_silver = df_silver \
    .withColumn("mes", F.month("lpep_pickup_datetime")) \
    .withColumn("dia_semana", F.dayofweek("lpep_pickup_datetime")) \
    .withColumn("hora", F.hour("lpep_pickup_datetime"))

# 3. Cruce espacial (Variedad)
df_silver_enriched = df_silver.join(
    df_bronze_zones, 
    df_silver.PULocationID == df_bronze_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "pickup_borough") \
 .withColumnRenamed("Zone", "pickup_zone")

print(f"Registros válidos tras limpieza en Silver: {df_silver_enriched.count()}")

# Habilitar SQL
df_silver_enriched.createOrReplaceTempView("taxis_silver")

--- SCRIPT 3: TRANSFORMACIÓN Y CAPA SILVER ---
Registros válidos tras limpieza en Silver: 1102796


In [5]:
print("--- SCRIPT 4: CAPA GOLD - MÉTRICA DE DEMANDA ---")
df_demanda = spark.sql("""
    SELECT dia_semana, hora, COUNT(*) as total_viajes
    FROM taxis_silver 
    GROUP BY dia_semana, hora 
    ORDER BY dia_semana, hora
""")
df_demanda.show(5)
# Guardando dentro de work/ para que lo veas en Windows
df_demanda.coalesce(1).write.mode("overwrite").csv("work/powerbi_demanda", header=True)
print("-> Exportado a work/powerbi_demanda/")

--- SCRIPT 4: CAPA GOLD - MÉTRICA DE DEMANDA ---
+----------+----+------------+
|dia_semana|hora|total_viajes|
+----------+----+------------+
|         1|   0|        4181|
|         1|   1|        3466|
|         1|   2|        2878|
|         1|   3|        2454|
|         1|   4|        1912|
+----------+----+------------+
only showing top 5 rows

-> Exportado a work/powerbi_demanda/


In [6]:
print("--- SCRIPT 5: CAPA GOLD - MÉTRICA DE RENTABILIDAD POR ZONA ---")
df_zonas = spark.sql("""
    SELECT pickup_borough as distrito, pickup_zone as zona, 
           COUNT(*) as total_viajes, ROUND(AVG(total_amount), 2) as ingreso_promedio_usd
    FROM taxis_silver 
    GROUP BY pickup_borough, pickup_zone 
    HAVING total_viajes > 500 
    ORDER BY ingreso_promedio_usd DESC
""")
df_zonas.show(5)
df_zonas.coalesce(1).write.mode("overwrite").csv("work/powerbi_zonas", header=True)
print("-> Exportado a work/powerbi_zonas/")

--- SCRIPT 5: CAPA GOLD - MÉTRICA DE RENTABILIDAD POR ZONA ---
+--------+--------------------+------------+--------------------+
|distrito|                zona|total_viajes|ingreso_promedio_usd|
+--------+--------------------+------------+--------------------+
|  Queens|       Willets Point|         529|               76.18|
|  Queens|Flushing Meadows-...|       10437|               68.98|
|Brooklyn|            Red Hook|        4094|               61.16|
|   Bronx|          Highbridge|         973|               60.44|
|  Queens|              Corona|         682|               57.49|
+--------+--------------------+------------+--------------------+
only showing top 5 rows

-> Exportado a work/powerbi_zonas/


In [7]:
print("--- SCRIPT 6: CAPA GOLD - MÉTRICA DE MÉTODOS DE PAGO ---")
df_pagos = spark.sql("""
    SELECT 
        CASE 
            WHEN payment_type = 1 THEN 'Tarjeta de Crédito'
            WHEN payment_type = 2 THEN 'Efectivo'
            WHEN payment_type = 3 THEN 'Sin Cargo'
            WHEN payment_type = 4 THEN 'Disputa'
            ELSE 'Otro'
        END as tipo_pago,
        COUNT(*) as cantidad_viajes
    FROM taxis_silver 
    GROUP BY tipo_pago
""")
df_pagos.show(5)
df_pagos.coalesce(1).write.mode("overwrite").csv("work/powerbi_pagos", header=True)
print("-> Exportado a work/powerbi_pagos/")

--- SCRIPT 6: CAPA GOLD - MÉTRICA DE MÉTODOS DE PAGO ---
+------------------+---------------+
|         tipo_pago|cantidad_viajes|
+------------------+---------------+
|              Otro|             38|
|Tarjeta de Crédito|         807372|
|           Disputa|           1178|
|         Sin Cargo|           3244|
|          Efectivo|         290964|
+------------------+---------------+

-> Exportado a work/powerbi_pagos/


In [8]:
print("--- SCRIPT 7: CAPA GOLD - MÉTRICA DE DISPERSIÓN ---")
df_dispersion = spark.sql("""
    SELECT trip_distance as distancia_millas, total_amount as tarifa_total, hora
    FROM taxis_silver 
    WHERE trip_distance < 30 AND total_amount < 150
""").sample(fraction=0.05, seed=42)

df_dispersion.show(5)
df_dispersion.coalesce(1).write.mode("overwrite").csv("work/powerbi_dispersion", header=True)
print("-> Exportado a work/powerbi_dispersion/")

--- SCRIPT 7: CAPA GOLD - MÉTRICA DE DISPERSIÓN ---
+----------------+------------+----+
|distancia_millas|tarifa_total|hora|
+----------------+------------+----+
|            1.19|        12.5|   0|
|            0.67|        7.92|   6|
|             7.8|       45.14|   6|
|            1.47|        10.4|   6|
|             0.6|         9.1|   7|
+----------------+------------+----+
only showing top 5 rows

-> Exportado a work/powerbi_dispersion/


In [9]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("--- SCRIPT 8: MODELO DE MACHINE LEARNING (REGRESIÓN LINEAL) ---")

df_ml = df_silver_enriched.select("trip_distance", "hora", "dia_semana", "total_amount").dropna()
ensamblador = VectorAssembler(inputCols=["trip_distance", "hora", "dia_semana"], outputCol="features")
df_ml_vector = ensamblador.transform(df_ml).select("features", "total_amount")

train_data, test_data = df_ml_vector.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="total_amount")
lr_modelo = lr.fit(train_data)

predicciones = lr_modelo.transform(test_data)
evaluador_rmse = RegressionEvaluator(labelCol="total_amount", predictionCol="prediction", metricName="rmse")
evaluador_r2 = RegressionEvaluator(labelCol="total_amount", predictionCol="prediction", metricName="r2")

print(f"Métricas del Modelo:")
print(f"- RMSE (Margen de error promedio): {evaluador_rmse.evaluate(predicciones):.2f} USD")
print(f"- R2 (Correlación): {evaluador_r2.evaluate(predicciones):.2f}")

--- SCRIPT 8: MODELO DE MACHINE LEARNING (REGRESIÓN LINEAL) ---
Métricas del Modelo:
- RMSE (Margen de error promedio): 20.33 USD
- R2 (Correlación): -0.16
